In [0]:
%sql
    
-- Setup: create schema and tables (idempotent)

CREATE SCHEMA IF NOT EXISTS incremental_demo;
USE incremental_demo;

DROP TABLE IF EXISTS bronze_customer;
DROP TABLE IF EXISTS silver_customer;
DROP TABLE IF EXISTS stg_changed_customer;
DROP TABLE IF EXISTS processing_watermark;

-- Bronze: append-only. No update_ts because rows are NEVER modified.
-- business_update_ts = when the change happened in the source system (business clock).
CREATE TABLE bronze_customer (
    customer_id        BIGINT,
    customer_name      STRING,
    city               STRING,
    business_update_ts TIMESTAMP,
    insert_ts          TIMESTAMP
) USING DELTA;

-- Silver: mutable. Has both insert_ts and update_ts (Silver's own clock)
-- plus business_update_ts to track which business version is current.
CREATE TABLE silver_customer (
    customer_id        BIGINT,
    customer_name      STRING,
    city               STRING,
    business_update_ts TIMESTAMP,
    insert_ts          TIMESTAMP,
    update_ts          TIMESTAMP
) USING DELTA;

-- Metadata table: one row per target, stores the watermark.
CREATE TABLE processing_watermark (
    table_name    STRING,
    watermark_ts  TIMESTAMP
) USING DELTA;

   
## Run 1: Initial Load

The watermark starts at `1900-01-01` so that all existing Bronze rows are treated as new on the first run.

**Key additions in this variant:**
- `business_update_ts` — the source-system timestamp indicating when the business event actually occurred (distinct from `insert_ts` which is when the row arrived in Bronze).
- The MERGE uses a **conditional update**: `WHEN MATCHED AND src.business_update_ts > tgt.business_update_ts`. This ensures out-of-order (stale) arrivals never overwrite a newer version already in Silver.

**Note:** We capture `run_start_ts` BEFORE staging. This is the value we'll use as the watermark after successful processing.

In [0]:
%sql
    
-- Seed Bronze with two customers and initialize the watermark
-- business_update_ts represents when the change occurred in the source system

INSERT INTO bronze_customer VALUES
    (1, 'Ania',   'Warszawa', '2026-05-01 10:00:00', current_timestamp()),
    (2, 'Marcin', 'Gdańsk',   '2026-05-01 11:00:00', current_timestamp());

INSERT INTO processing_watermark VALUES
    ('silver_customer', to_timestamp('1900-01-01 00:00:00'));

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- Step 1: Capture processing start time BEFORE anything else.
-- This is our watermark value — a universal clock reference.

DECLARE OR REPLACE VARIABLE run_start_ts TIMESTAMP = current_timestamp();

SELECT run_start_ts AS captured_start_time;

captured_start_time
2026-05-12T14:04:29.905Z


In [0]:
%sql
-- Step 2: Stage changed rows.
-- Staging freezes the batch for debuggability and MERGE consistency.

CREATE OR REPLACE TABLE stg_changed_customer
USING DELTA
AS
SELECT b.*
FROM bronze_customer b
JOIN processing_watermark w
    ON w.table_name = 'silver_customer'
WHERE b.insert_ts > w.watermark_ts;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Inspect what will be processed in this run
SELECT * FROM stg_changed_customer ORDER BY customer_id, insert_ts;

customer_id,customer_name,city,business_update_ts,insert_ts
1,Ania,Warszawa,2026-05-01T10:00:00.000Z,2026-05-12T14:04:02.927Z
2,Marcin,Gdańsk,2026-05-01T11:00:00.000Z,2026-05-12T14:04:02.927Z


In [0]:
%sql
    
-- Step 3: MERGE into Silver
-- Deduplicate: keep latest Bronze row per business key (by business_update_ts)
-- Conditional update: only apply if incoming business version is NEWER than what Silver has.
-- This protects against out-of-order arrivals.

MERGE INTO silver_customer AS tgt
USING (
    SELECT
        customer_id,
        customer_name,
        city,
        business_update_ts,
        current_timestamp() AS processing_ts
    FROM stg_changed_customer
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY business_update_ts DESC
    ) = 1
) AS src
ON tgt.customer_id = src.customer_id

WHEN MATCHED AND src.business_update_ts > tgt.business_update_ts THEN UPDATE SET
    tgt.customer_name      = src.customer_name,
    tgt.city               = src.city,
    tgt.business_update_ts = src.business_update_ts,
    tgt.update_ts          = src.processing_ts

WHEN NOT MATCHED THEN INSERT (
    customer_id, customer_name, city, business_update_ts, insert_ts, update_ts
) VALUES (
    src.customer_id, src.customer_name, src.city, src.business_update_ts, src.processing_ts, src.processing_ts
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
-- Step 4: Advance watermark to run_start_ts

UPDATE processing_watermark
SET watermark_ts = run_start_ts
WHERE table_name = 'silver_customer';

num_affected_rows
1


In [0]:
%sql
-- Verify: Silver should contain current state
SELECT * FROM silver_customer ORDER BY customer_id;

customer_id,customer_name,city,business_update_ts,insert_ts,update_ts
1,Ania,Warszawa,2026-05-01T10:00:00.000Z,2026-05-12T14:05:28.207Z,2026-05-12T14:05:28.207Z
2,Marcin,Gdańsk,2026-05-01T11:00:00.000Z,2026-05-12T14:05:28.207Z,2026-05-12T14:05:28.207Z


In [0]:
%sql
SELECT * FROM processing_watermark;

   
## Run 2: Incremental Processing

Now we simulate two source-side changes:
- A **new customer** (Tomek) appears with `business_update_ts = 2026-05-02 09:00:00`.
- An **existing customer** (Marcin) changes city from Gdańsk to Wrocław with `business_update_ts = 2026-05-02 14:00:00`.

Because Bronze is append-only, Marcin's update is represented as a **new row** — the old row remains untouched.

**Watch the pattern:** Same four steps — capture start_ts, stage, MERGE, advance watermark.

In [0]:
%sql
    
-- Append new Bronze events (Bronze is NEVER updated in place)
-- business_update_ts reflects the actual source-system event time

-- New customer
INSERT INTO bronze_customer VALUES
    (3, 'Tomek', 'Kraków', '2026-05-02 09:00:00', current_timestamp());

-- Marcin changed city: represented as a NEW Bronze row with a newer business_update_ts
INSERT INTO bronze_customer VALUES
    (2, 'Marcin', 'Wrocław', '2026-05-02 14:00:00', current_timestamp());

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
    
-- Bronze keeps full history: both versions of Marcin are visible
SELECT * FROM bronze_customer ORDER BY customer_id, insert_ts;

customer_id,customer_name,city,business_update_ts,insert_ts
1,Ania,Warszawa,2026-05-01T10:00:00.000Z,2026-05-12T14:04:02.927Z
2,Marcin,Gdańsk,2026-05-01T11:00:00.000Z,2026-05-12T14:04:02.927Z
2,Marcin,Wrocław,2026-05-02T14:00:00.000Z,2026-05-12T14:06:50.836Z
3,Tomek,Kraków,2026-05-02T09:00:00.000Z,2026-05-12T14:06:48.998Z


In [0]:
%sql
-- Step 1 (Run 2): Capture new start time

SET VAR run_start_ts = current_timestamp();

SELECT run_start_ts AS captured_start_time_run2;

captured_start_time_run2
2026-05-12T14:07:46.895Z


In [0]:
%sql
-- Step 2 (Run 2): Only rows newer than the watermark are detected

CREATE OR REPLACE TABLE stg_changed_customer
USING DELTA
AS
SELECT b.*
FROM bronze_customer b
JOIN processing_watermark w
    ON w.table_name = 'silver_customer'
WHERE b.insert_ts > w.watermark_ts;

num_affected_rows,num_inserted_rows


In [0]:
%sql
    
-- Only Marcin (new version) and Tomek are detected — Ania is NOT re-read
SELECT * FROM stg_changed_customer ORDER BY customer_id, insert_ts;

customer_id,customer_name,city,business_update_ts,insert_ts
2,Marcin,Wrocław,2026-05-02T14:00:00.000Z,2026-05-12T14:06:50.836Z
3,Tomek,Kraków,2026-05-02T09:00:00.000Z,2026-05-12T14:06:48.998Z


In [0]:
%sql
    
-- Step 3 (Run 2): Same MERGE logic — idempotent by design
-- Conditional update skips rows with older business_update_ts

MERGE INTO silver_customer AS tgt
USING (
    SELECT
        customer_id,
        customer_name,
        city,
        business_update_ts,
        current_timestamp() AS processing_ts
    FROM stg_changed_customer
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY business_update_ts DESC
    ) = 1
) AS src
ON tgt.customer_id = src.customer_id

WHEN MATCHED AND src.business_update_ts > tgt.business_update_ts THEN UPDATE SET
    tgt.customer_name      = src.customer_name,
    tgt.city               = src.city,
    tgt.business_update_ts = src.business_update_ts,
    tgt.update_ts          = src.processing_ts

WHEN NOT MATCHED THEN INSERT (
    customer_id, customer_name, city, business_update_ts, insert_ts, update_ts
) VALUES (
    src.customer_id, src.customer_name, src.city, src.business_update_ts, src.processing_ts, src.processing_ts
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
-- Step 4 (Run 2): Advance watermark to start_time

UPDATE processing_watermark
SET watermark_ts = run_start_ts
WHERE table_name = 'silver_customer';

num_affected_rows
1


In [0]:
%sql
    
-- Final state: Ania unchanged, Marcin updated to Wrocław, Tomek added
SELECT * FROM silver_customer ORDER BY customer_id;

customer_id,customer_name,city,business_update_ts,insert_ts,update_ts
1,Ania,Warszawa,2026-05-01T10:00:00.000Z,2026-05-12T14:05:28.207Z,2026-05-12T14:05:28.207Z
2,Marcin,Wrocław,2026-05-02T14:00:00.000Z,2026-05-12T14:05:28.207Z,2026-05-12T14:08:18.582Z
3,Tomek,Kraków,2026-05-02T09:00:00.000Z,2026-05-12T14:08:18.582Z,2026-05-12T14:08:18.582Z


In [0]:
%sql
SELECT * FROM processing_watermark;

   
## Run 3: Out-of-Order Arrival (Late-Arriving Old Version)

Now we simulate a **late-arriving stale event**: Marcin's OLD address (Gdańsk) arrives in Bronze AFTER his newer address (Wrocław) was already processed.

**Expected outcome:** The MERGE detects that Silver already holds `business_update_ts = 2026-05-02 14:00:00` for Marcin, which is NEWER than the incoming `2026-05-01 12:00:00`. The conditional `WHEN MATCHED AND src.business_update_ts > tgt.business_update_ts` is FALSE, so **Marcin is NOT updated** — the stale row is silently skipped.

In [0]:
%sql
    
-- Simulate an out-of-order OLD version of Marcin.
-- business_update_ts = 2026-05-01 12:00:00, which is OLDER than what Silver has (2026-05-02 14:00:00).

INSERT INTO bronze_customer VALUES
    (2, 'Marcin', 'Gdańsk', '2026-05-01 12:00:00', current_timestamp());

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
    
-- Bronze now has THREE rows for Marcin: the old version arrived last
SELECT * FROM bronze_customer WHERE customer_id = 2 ORDER BY insert_ts;

customer_id,customer_name,city,business_update_ts,insert_ts
2,Marcin,Gdańsk,2026-05-01T11:00:00.000Z,2026-05-12T14:04:02.927Z
2,Marcin,Wrocław,2026-05-02T14:00:00.000Z,2026-05-12T14:06:50.836Z
2,Marcin,Gdańsk,2026-05-01T12:00:00.000Z,2026-05-12T14:09:34.595Z


In [0]:
%sql
    
-- Step 1 (Run 3): Capture new start time

SET VAR run_start_ts = current_timestamp();

SELECT run_start_ts AS captured_start_time_run3;

captured_start_time_run3
2026-05-12T14:11:11.902Z


In [0]:
%sql
    
-- Step 2 (Run 3): Stage rows newer than watermark

CREATE OR REPLACE TABLE stg_changed_customer
USING DELTA
AS
SELECT b.*
FROM bronze_customer b
JOIN processing_watermark w
    ON w.table_name = 'silver_customer'
WHERE b.insert_ts > w.watermark_ts;

num_affected_rows,num_inserted_rows


In [0]:
%sql
    
-- Only the late-arriving Marcin row is staged (it has a recent insert_ts)
-- Note: its business_update_ts is OLD (2026-05-01 12:00:00)
SELECT * FROM stg_changed_customer ORDER BY customer_id, insert_ts;

customer_id,customer_name,city,business_update_ts,insert_ts
2,Marcin,Gdańsk,2026-05-01T12:00:00.000Z,2026-05-12T14:09:34.595Z


In [0]:
%sql
    
-- Step 3 (Run 3): MERGE with out-of-order protection
-- The WHEN MATCHED condition checks: src.business_update_ts > tgt.business_update_ts
-- Silver has Marcin at 2026-05-02 14:00:00 > incoming 2026-05-01 12:00:00
-- Therefore: Marcin is NOT updated. The stale row is skipped.

MERGE INTO silver_customer AS tgt
USING (
    SELECT
        customer_id,
        customer_name,
        city,
        business_update_ts,
        current_timestamp() AS processing_ts
    FROM stg_changed_customer
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY business_update_ts DESC
    ) = 1
) AS src
ON tgt.customer_id = src.customer_id

WHEN MATCHED AND src.business_update_ts > tgt.business_update_ts THEN UPDATE SET
    tgt.customer_name      = src.customer_name,
    tgt.city               = src.city,
    tgt.business_update_ts = src.business_update_ts,
    tgt.update_ts          = src.processing_ts

WHEN NOT MATCHED THEN INSERT (
    customer_id, customer_name, city, business_update_ts, insert_ts, update_ts
) VALUES (
    src.customer_id, src.customer_name, src.city, src.business_update_ts, src.processing_ts, src.processing_ts
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
    
-- Step 4 (Run 3): Advance watermark

UPDATE processing_watermark
SET watermark_ts = run_start_ts
WHERE table_name = 'silver_customer';

num_affected_rows
1


In [0]:
%sql
    
-- VERIFY: Marcin is STILL in Wrocław (2026-05-02 14:00:00), NOT reverted to Gdańsk.
-- The out-of-order old version was correctly skipped.
SELECT * FROM silver_customer ORDER BY customer_id;

In [0]:
%sql
    
SELECT * FROM processing_watermark;

   
## Atomicity and Idempotency

The MERGE and watermark UPDATE are two separate statements. On **Databricks**, you CAN wrap them in a single atomic transaction:

```sql
BEGIN ATOMIC
  MERGE INTO silver_customer AS tgt USING ... ;
  UPDATE processing_watermark SET watermark_ts = run_start_ts ... ;
END;
```

This gives you all-or-nothing semantics — if either statement fails, both are rolled back.

**However**, we intentionally design the pattern to be safe WITHOUT transactions too (for portability):

| Failure scenario | What happens on next run | Result |
| --- | --- | --- |
| Crash **before** staging | Nothing processed. Watermark unchanged. Next run retries. | Safe |
| Crash **after** MERGE but **before** watermark update | Watermark still at old value. Next run re-stages same rows, re-runs same MERGE. Idempotent → same result. | Safe |
| Crash **after** watermark update | Complete success. | Safe |

**The takeaway:**
- On Databricks: use `BEGIN ATOMIC ... END;` for guaranteed atomicity.
- On platforms without multi-table transactions: the pattern is still crash-safe because the MERGE is idempotent.
- Start_time + crash replay = harmless: some rows get reprocessed, but the conditional `business_update_ts` check prevents stale data from overwriting newer versions.
- Out-of-order replay is also safe: even if a stale row is staged again, the MERGE condition rejects it.

   
## Key Takeaways

* **Staging + start_time** is the recommended combination: staging for batch freeze and debuggability, start_time for simplicity.
* **`business_update_ts`** is the source-system event timestamp — it answers "when did this change actually happen?" as opposed to `insert_ts` which answers "when did Bronze receive it?"
* **Conditional MERGE** (`WHEN MATCHED AND src.business_update_ts > tgt.business_update_ts`) protects Silver from being overwritten by stale, out-of-order arrivals.
* The watermark always advances.
* One watermark per target suffices even for multi-source targets (start_time is a universal clock reference).
* Staging is decoupled from watermark calculation — you don't need `MAX(insert_ts)` from staging.
* The MERGE is idempotent, so crash replay (re-applying the same batch) is harmless.
* For concurrent writers: add a lookback window (`>= watermark_ts - INTERVAL 10 MINUTES`) to protect against timestamps generated before commit.
* This is a **detection** pattern. How to apply detected changes (SCD1, SCD2, aggregates) is a separate concern.